<a href="https://colab.research.google.com/github/camilavazquezz/colab/blob/main/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [14]:
# Load the King County House Sales dataset
# Data source: Kaggle - House Sales in King County, USA
# https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

df = pd.read_csv("kc_house_data.csv")

# Display the first 5 houses
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


Data source: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction?resource=download


In [15]:
print("Number of houses:", len(df))
print("\nColumns:")
print(df.columns.tolist())

Number of houses: 21613

Columns:
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


In [16]:
# Select the features needed for house price prediction
house_data = df[['sqft_living', 'zipcode', 'price']].copy()

# Rename columns to make them easier to understand
house_data = house_data.rename(columns={
    'sqft_living': 'square_footage',
    'zipcode': 'location'
})

# Display the first 10 houses
house_data.head(10)

,square_footage,location,price
0,1180,98178,221900.0
1,2570,98125,538000.0
2,770,98028,180000.0
3,1960,98136,604000.0
4,1680,98074,510000.0
5,5420,98053,1225000.0
6,1715,98003,257500.0
7,1060,98198,291850.0
8,1780,98146,229500.0
9,1890,98038,323000.0


In [17]:
# Features used to predict the house price
X = house_data[['square_footage', 'location']]

# Target variable (what we want to predict)
y = house_data['price']

print("Features:")
print(X.head())

print("\nTarget prices:")
print(y.head())

Features:
   square_footage  location
0            1180     98178
1            2570     98125
2             770     98028
3            1960     98136
4            1680     98074

Target prices:
0    221900.0
1    538000.0
2    180000.0
3    604000.0
4    510000.0
Name: price, dtype: float64


In [18]:
# Treat ZIP code as a categorical location
X = X.copy()
X['location'] = X['location'].astype(str)

print(X.head())
print("\nData types:")
print(X.dtypes)

   square_footage location
0            1180    98178
1            2570    98125
2             770    98028
3            1960    98136
4            1680    98074

Data types:
square_footage     int64
location          object
dtype: object


In [19]:
# Preprocess the location column using OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('location', OneHotEncoder(handle_unknown='ignore'), ['location'])
    ],
    remainder='passthrough'
)

# Create the Linear Regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Split the real housing data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print("Training houses:", len(X_train))
print("Testing houses:", len(X_test))

Model trained successfully!
Training houses: 17290
Testing houses: 4323


In [20]:
# Predict the price of a 2000 sq ft house in ZIP code 98125
new_house = pd.DataFrame({
    'square_footage': [2000],
    'location': ['98125']
})

predicted_price = model.predict(new_house)

print(f"Predicted price for a 2000 sq ft house in ZIP code 98125: ${predicted_price[0]:,.2f}")

Predicted price for a 2000 sq ft house in ZIP code 98125: $532,911.65


In [21]:
# Get the model coefficients
coefficients = model.named_steps['regressor'].coef_

# Get the feature names after OneHotEncoding
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

# Find and print the square footage coefficient
for feature, coef in zip(feature_names, coefficients):
    if 'square_footage' in feature:
        print(f"Square footage coefficient: ${coef:,.2f} per square foot")

Square footage coefficient: $251.00 per square foot


In [22]:
# Display some location (ZIP code) effects
print("Sample Location Effects:")

count = 0

for feature, coef in zip(feature_names, coefficients):
    if 'location' in feature:
        print(f"{feature}: ${coef:,.2f}")
        count += 1

    if count == 10:
        break

Sample Location Effects:
location__location_98001: $-225,593.35
location__location_98002: $-204,459.71
location__location_98003: $-227,246.45
location__location_98004: $574,795.75
location__location_98005: $98,009.91
location__location_98006: $114,550.73
location__location_98007: $33,380.98
location__location_98008: $69,642.28
location__location_98010: $-150,903.16
location__location_98011: $-106,338.79
